# 25 · Model Interpretation & Error Analysis

**2026 learning module** · Run cells from top to bottom. This notebook uses built-in/sample datasets and fixed random seeds where appropriate.

## Goal
Inspect a model with permutation importance, a confusion matrix, and a precision-recall curve. Importance is measured on held-out data, not on the training set.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay, classification_report

data = load_breast_cancer(as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.25, stratify=data.target, random_state=21)
model = RandomForestClassifier(n_estimators=300, random_state=21, class_weight="balanced")
model.fit(X_train, y_train)
pred = model.predict(X_test)
print(classification_report(y_test, pred, target_names=data.target_names))

In [ ]:
result = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=21, scoring="f1", n_jobs=-1)
importance = pd.Series(result.importances_mean, index=X_test.columns).sort_values().tail(15)
importance.plot(kind="barh", figsize=(8, 6), title="Top permutation importances (held-out data)");
plt.xlabel("Mean decrease in F1 when feature is shuffled"); plt.tight_layout()

In [ ]:
ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, display_labels=data.target_names, xticks_rotation=25);
plt.tight_layout()
PrecisionRecallDisplay.from_estimator(model, X_test, y_test);
plt.tight_layout()

## Interpret with care
Permutation importance can be misleading when features are strongly correlated, and it describes this model on this dataset—not causal effects. For consequential applications, add domain review and external validation.